In [1]:
# ============================================================
# 13_ROMA_AURORA_unified_comparison.ipynb
# Unified ROMA-AURORA Final Paper Tables and Figures
#
# Purpose:
# 1. Load ROMA R3 outputs and AURORA Notebook 10/11/12 outputs.
# 2. Build unified strict-test comparison tables.
# 3. Compare ROMA, AURORA, and strong benchmarks on identical dates.
# 4. Summarize statistical evidence.
# 5. Export final paper-ready tables, figures, captions, and manuscript text.
#
# Main interpretation:
# - ROMA = leakage-controlled regime-template baseline.
# - AURORA10-UAMV-B = uncertainty-aware allocation method.
# - B12 = strong moving-average timing benchmark.
#
# Educational/research use only.
# Not personalized financial advice.
# ============================================================

from __future__ import annotations

import json
import math
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ============================================================
# 0. Colab setup
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False

# ============================================================
# 1. Paths and run configuration
# ============================================================

PROJECT_CODE = "ROMA_AURORA_TWETF"
ROMA_CODE = "ROMA_TWETF"
AURORA_CODE = "AURORA_TWETF"

PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

# Completed ROMA R3 run.
ROMA_R3_RUN_ID = "20260625_031440"

ROMA_R3_ROOT = (
    PUBLICATION_ROOT
    / "outputs"
    / ROMA_CODE
    / "statistical_significance_block_bootstrap"
    / f"run_{ROMA_R3_RUN_ID}"
)

NOTEBOOK13_INPUT_INDEX_PATH = ROMA_R3_ROOT / "NOTEBOOK13_ROMA_AURORA_INPUT_INDEX.csv"

# AURORA finalized paper-assets run.
# Used when available for Notebook 12 final tables.
AURORA_NOTEBOOK12_RUN_ID = "20260624_142307"
AURORA_NOTEBOOK12_ROOT = (
    PUBLICATION_ROOT
    / "outputs"
    / AURORA_CODE
    / "paper_tables_figures_manuscript_assets"
    / f"run_{AURORA_NOTEBOOK12_RUN_ID}"
)

# AURORA statistical run.
AURORA_NOTEBOOK11_RUN_ID = "20260624_124834"
AURORA_NOTEBOOK11_ROOT = (
    PUBLICATION_ROOT
    / "outputs"
    / AURORA_CODE
    / "statistical_significance_block_bootstrap"
    / f"run_{AURORA_NOTEBOOK11_RUN_ID}"
)

# AURORA allocation run.
AURORA_NOTEBOOK10_RUN_ID = "20260624_100748"
AURORA_NOTEBOOK10_ROOT = (
    PUBLICATION_ROOT
    / "outputs"
    / AURORA_CODE
    / "uncertainty_aware_mean_variance_allocation"
    / f"run_{AURORA_NOTEBOOK10_RUN_ID}"
)

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"
FIGURE_DIR = OUTPUT_ROOT / "figures"
MANUSCRIPT_DIR = OUTPUT_ROOT / "manuscript_assets"

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

RUN_ROOT = OUTPUT_ROOT / "unified_paper_comparison" / f"run_{RUN_ID}"

TABLE_RUN_DIR = RUN_ROOT / "tables"
REPORT_RUN_DIR = RUN_ROOT / "reports"
FIGURE_RUN_DIR = RUN_ROOT / "figures"
PAPER_FIGURE_DIR = RUN_ROOT / "paper_figures"
MANUSCRIPT_RUN_DIR = RUN_ROOT / "manuscript_assets"
RETURN_DIR = RUN_ROOT / "returns"

for d in [
    OUTPUT_ROOT,
    TABLE_DIR,
    REPORT_DIR,
    FIGURE_DIR,
    MANUSCRIPT_DIR,
    RUN_ROOT,
    TABLE_RUN_DIR,
    REPORT_RUN_DIR,
    FIGURE_RUN_DIR,
    PAPER_FIGURE_DIR,
    MANUSCRIPT_RUN_DIR,
    RETURN_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("Notebook 13: Unified ROMA-AURORA Final Comparison")
print("=" * 80)
print("Timestamp UTC            :", RUN_TIMESTAMP)
print("Run ID                   :", RUN_ID)
print("ROMA R3 run ID           :", ROMA_R3_RUN_ID)
print("Notebook13 input index   :", NOTEBOOK13_INPUT_INDEX_PATH)
print("AURORA Notebook10 root   :", AURORA_NOTEBOOK10_ROOT)
print("AURORA Notebook11 root   :", AURORA_NOTEBOOK11_ROOT)
print("AURORA Notebook12 root   :", AURORA_NOTEBOOK12_ROOT)
print("Run root                 :", RUN_ROOT)
print("=" * 80)

if not NOTEBOOK13_INPUT_INDEX_PATH.exists():
    raise FileNotFoundError(f"Missing Notebook 13 input index: {NOTEBOOK13_INPUT_INDEX_PATH}")

# ============================================================
# 2. Global settings
# ============================================================

ANNUALIZATION = 252

PRIMARY_SPLIT_NAME = "strict_test_only"

PRIMARY_ROMA_POLICY = "ROMA_P4_balanced_regime_blend"
PRIMARY_AURORA_POLICY = "AURORA10_UAMV_B_more60_defensive"
VALIDATION_SELECTED_AURORA_POLICY = "AURORA10_validation_selected_UAMV"

PRIMARY_BENCHMARKS = [
    "B12_ma_timing_equal_weight",
    "B6_00881_only",
    "B3_0050_only",
    "B1_equal_weight_all_etfs",
    "B10_momentum_top2_63d",
    "B16_minimum_variance_126d",
    "B15_minimum_variance_126d",
]

FINAL_POLICY_ORDER = [
    PRIMARY_AURORA_POLICY,
    "AURORA10_UAMV_D_low_turnover",
    "AURORA10_UAMV_E_no_regime_tilt_control",
    "AURORA10_UAMV_A_balanced",
    "AURORA10_UAMV_C_more60_growth",
    VALIDATION_SELECTED_AURORA_POLICY,
    PRIMARY_ROMA_POLICY,
    "ROMA_P2_20d_only_return_seeking",
    "ROMA_P0_validation_selected_20_60_blend",
    "B12_ma_timing_equal_weight",
    "B6_00881_only",
    "B3_0050_only",
    "B1_equal_weight_all_etfs",
    "B10_momentum_top2_63d",
    "B16_minimum_variance_126d",
    "B15_minimum_variance_126d",
]

POLICY_DISPLAY_NAMES = {
    "AURORA10_UAMV_B_more60_defensive": "AURORA10-UAMV-B",
    "AURORA10_UAMV_D_low_turnover": "AURORA10-UAMV-D",
    "AURORA10_UAMV_E_no_regime_tilt_control": "AURORA10-UAMV-E",
    "AURORA10_UAMV_A_balanced": "AURORA10-UAMV-A",
    "AURORA10_UAMV_C_more60_growth": "AURORA10-UAMV-C",
    "AURORA10_validation_selected_UAMV": "AURORA10 validation-selected UAMV",
    "ROMA_P4_balanced_regime_blend": "ROMA-P4 balanced regime",
    "ROMA_P2_20d_only_return_seeking": "ROMA-P2 20d return-seeking",
    "ROMA_P0_validation_selected_20_60_blend": "ROMA-P0 20/60 blend",
    "B12_ma_timing_equal_weight": "B12 MA timing equal-weight",
    "B6_00881_only": "B6 00881 only",
    "B3_0050_only": "B3 0050 only",
    "B1_equal_weight_all_etfs": "B1 equal-weight all ETFs",
    "B10_momentum_top2_63d": "B10 momentum top-2",
    "B16_minimum_variance_126d": "B16 minimum variance",
    "B15_minimum_variance_126d": "B15 minimum variance",
}

POLICY_GROUPS = {
    "AURORA10_UAMV_B_more60_defensive": "AURORA uncertainty-aware allocation",
    "AURORA10_UAMV_D_low_turnover": "AURORA uncertainty-aware allocation",
    "AURORA10_UAMV_E_no_regime_tilt_control": "AURORA uncertainty-aware allocation",
    "AURORA10_UAMV_A_balanced": "AURORA uncertainty-aware allocation",
    "AURORA10_UAMV_C_more60_growth": "AURORA uncertainty-aware allocation",
    "AURORA10_validation_selected_UAMV": "AURORA validation-selected",
    "ROMA_P4_balanced_regime_blend": "ROMA regime-template baseline",
    "ROMA_P2_20d_only_return_seeking": "ROMA regime-template baseline",
    "ROMA_P0_validation_selected_20_60_blend": "ROMA regime-template baseline",
    "B12_ma_timing_equal_weight": "technical benchmark",
    "B6_00881_only": "single-ETF benchmark",
    "B3_0050_only": "single-ETF benchmark",
    "B1_equal_weight_all_etfs": "passive benchmark",
    "B10_momentum_top2_63d": "technical benchmark",
    "B16_minimum_variance_126d": "risk-based benchmark",
    "B15_minimum_variance_126d": "risk-based benchmark",
}

# ============================================================
# 3. Utility functions
# ============================================================

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)

    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []

    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })

    return pd.DataFrame(rows)

def safe_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(" ", "_")
        .replace(".", "_")
        .replace("%", "pct")
        .replace("-", "_")
        .replace("+", "plus")
    )

def read_table(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(path)

    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)

    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)

    raise ValueError(f"Unsupported file type: {path}")

def maybe_read_table(path):
    path = Path(path)
    if path.exists():
        return read_table(path)
    return None

def calculate_drawdown(equity):
    equity = pd.Series(equity).astype(float)
    running_max = equity.cummax()
    return equity / running_max - 1.0

def performance_metrics(r, annualization=ANNUALIZATION):
    r = pd.Series(r).dropna().astype(float)

    if len(r) == 0:
        return {
            "n_days": 0,
            "start_date": pd.NaT,
            "end_date": pd.NaT,
            "total_return": np.nan,
            "annual_return": np.nan,
            "annual_volatility": np.nan,
            "sharpe": np.nan,
            "sortino": np.nan,
            "max_drawdown": np.nan,
            "calmar": np.nan,
            "final_equity": np.nan,
            "mean_daily_return": np.nan,
            "daily_volatility": np.nan,
            "positive_day_rate": np.nan,
            "worst_daily_return": np.nan,
            "best_daily_return": np.nan,
        }

    r.index = pd.to_datetime(r.index)
    equity = (1.0 + r).cumprod()
    total_return = float(equity.iloc[-1] - 1.0)

    n = len(r)
    annual_return = float(equity.iloc[-1] ** (annualization / n) - 1.0)

    daily_vol = float(r.std(ddof=1)) if n > 1 else np.nan
    annual_vol = float(daily_vol * np.sqrt(annualization)) if np.isfinite(daily_vol) else np.nan

    mean_daily = float(r.mean())
    sharpe = float((mean_daily / daily_vol) * np.sqrt(annualization)) if daily_vol and daily_vol > 0 else np.nan

    downside = r[r < 0]
    downside_vol = float(downside.std(ddof=1)) if len(downside) > 1 else np.nan
    sortino = float((mean_daily / downside_vol) * np.sqrt(annualization)) if np.isfinite(downside_vol) and downside_vol > 0 else np.nan

    drawdown = calculate_drawdown(equity)
    max_drawdown = float(drawdown.min())

    calmar = float(annual_return / abs(max_drawdown)) if max_drawdown < 0 else np.nan

    return {
        "n_days": int(n),
        "start_date": r.index.min(),
        "end_date": r.index.max(),
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "sharpe": sharpe,
        "sortino": sortino,
        "max_drawdown": max_drawdown,
        "calmar": calmar,
        "final_equity": float(equity.iloc[-1]),
        "mean_daily_return": mean_daily,
        "daily_volatility": daily_vol,
        "positive_day_rate": float((r > 0).mean()),
        "worst_daily_return": float(r.min()),
        "best_daily_return": float(r.max()),
    }

def build_performance_table(return_matrix, policy_group_map=None):
    rows = []

    for policy in return_matrix.columns:
        r = return_matrix[policy].dropna()
        m = performance_metrics(r)
        m["policy_name"] = policy
        m["display_name"] = POLICY_DISPLAY_NAMES.get(policy, policy)
        m["policy_group"] = (
            policy_group_map.get(policy, POLICY_GROUPS.get(policy, "other"))
            if policy_group_map is not None
            else POLICY_GROUPS.get(policy, "other")
        )
        rows.append(m)

    out = pd.DataFrame(rows)

    if out.empty:
        return out

    out["rank_total_return"] = out["total_return"].rank(ascending=False, method="min")
    out["rank_annual_return"] = out["annual_return"].rank(ascending=False, method="min")
    out["rank_sharpe"] = out["sharpe"].rank(ascending=False, method="min")
    out["rank_sortino"] = out["sortino"].rank(ascending=False, method="min")
    out["rank_max_drawdown"] = out["max_drawdown"].rank(ascending=False, method="min")
    out["rank_calmar"] = out["calmar"].rank(ascending=False, method="min")

    out["composite_rank"] = (
        out["rank_total_return"]
        + out["rank_annual_return"]
        + out["rank_sharpe"]
        + out["rank_sortino"]
        + out["rank_max_drawdown"]
        + out["rank_calmar"]
    ) / 6.0

    out = out.sort_values(
        ["composite_rank", "sharpe", "total_return"],
        ascending=[True, False, False],
    ).reset_index(drop=True)

    return out

def paired_difference_metrics(strategy_returns, benchmark_returns):
    s = pd.Series(strategy_returns).dropna().astype(float)
    b = pd.Series(benchmark_returns).dropna().astype(float)

    common = s.index.intersection(b.index).sort_values()
    s = s.loc[common]
    b = b.loc[common]

    ps = performance_metrics(s)
    pb = performance_metrics(b)

    excess = s - b

    out = {
        "n_days": int(len(common)),
        "strategy_total_return": ps["total_return"],
        "benchmark_total_return": pb["total_return"],
        "diff_total_return": ps["total_return"] - pb["total_return"],
        "strategy_annual_return": ps["annual_return"],
        "benchmark_annual_return": pb["annual_return"],
        "diff_annual_return": ps["annual_return"] - pb["annual_return"],
        "strategy_annual_volatility": ps["annual_volatility"],
        "benchmark_annual_volatility": pb["annual_volatility"],
        "diff_annual_volatility": ps["annual_volatility"] - pb["annual_volatility"],
        "strategy_sharpe": ps["sharpe"],
        "benchmark_sharpe": pb["sharpe"],
        "diff_sharpe": ps["sharpe"] - pb["sharpe"],
        "strategy_sortino": ps["sortino"],
        "benchmark_sortino": pb["sortino"],
        "diff_sortino": ps["sortino"] - pb["sortino"],
        "strategy_max_drawdown": ps["max_drawdown"],
        "benchmark_max_drawdown": pb["max_drawdown"],
        "drawdown_improvement": ps["max_drawdown"] - pb["max_drawdown"],
        "strategy_calmar": ps["calmar"],
        "benchmark_calmar": pb["calmar"],
        "diff_calmar": ps["calmar"] - pb["calmar"],
        "annualized_mean_excess_return": float(excess.mean() * ANNUALIZATION),
        "excess_return_hit_rate": float((excess > 0).mean()),
        "mean_daily_excess_return": float(excess.mean()),
        "daily_tracking_error": float(excess.std(ddof=1)),
    }

    return out

def save_table(df, local_name, global_name):
    local_path = TABLE_RUN_DIR / local_name
    global_path = TABLE_DIR / global_name

    df.to_csv(local_path, index=False)
    df.to_csv(global_path, index=False)

    return local_path, global_path

def save_markdown(path, text):
    Path(path).write_text(text, encoding="utf-8")

def make_equity(return_series):
    r = pd.Series(return_series).dropna().astype(float)
    return (1.0 + r).cumprod()

def make_drawdown(return_series):
    eq = make_equity(return_series)
    return calculate_drawdown(eq)

# ============================================================
# 4. Load Notebook 13 input index
# ============================================================

print("\n" + "=" * 80)
print("Step 1: Loading Notebook 13 input index")
print("=" * 80)

nb13_index = pd.read_csv(NOTEBOOK13_INPUT_INDEX_PATH)

if nb13_index.empty:
    raise ValueError("Notebook 13 input index is empty.")

nb13 = nb13_index.iloc[0].to_dict()

print("Notebook 13 index:")
print(nb13_index.to_string(index=False))

roma_test_matrix_path = Path(nb13["roma_test_return_matrix_path"])
roma_oos_matrix_path = Path(nb13["roma_oos_return_matrix_path"])

aurora_test_matrix_path = (
    Path(nb13["aurora_test_return_matrix_path"])
    if pd.notna(nb13.get("aurora_test_return_matrix_path"))
    else None
)

roma_pairwise_path = Path(nb13["observed_pairwise_differences_path"])
roma_decision_table_path = Path(nb13["paper_decision_table_path"])
roma_bootstrap_summary_path = Path(nb13["primary_bootstrap_summary_path"])
roma_diagnostic_summary_path = Path(nb13["diagnostic_summary_path"])

# ============================================================
# 5. Load ROMA and AURORA matrices/tables
# ============================================================

print("\n" + "=" * 80)
print("Step 2: Loading ROMA and AURORA matrices/tables")
print("=" * 80)

roma_test_mat = read_table(roma_test_matrix_path)
roma_oos_mat = read_table(roma_oos_matrix_path)

if "date" in roma_test_mat.columns:
    roma_test_mat["date"] = pd.to_datetime(roma_test_mat["date"])
    roma_test_mat = roma_test_mat.set_index("date")
else:
    roma_test_mat.index = pd.to_datetime(roma_test_mat.index)

if "date" in roma_oos_mat.columns:
    roma_oos_mat["date"] = pd.to_datetime(roma_oos_mat["date"])
    roma_oos_mat = roma_oos_mat.set_index("date")
else:
    roma_oos_mat.index = pd.to_datetime(roma_oos_mat.index)

roma_test_mat = roma_test_mat.sort_index()
roma_oos_mat = roma_oos_mat.sort_index()

if aurora_test_matrix_path is not None and aurora_test_matrix_path.exists():
    aurora_test_mat = read_table(aurora_test_matrix_path)

    if "date" in aurora_test_mat.columns:
        aurora_test_mat["date"] = pd.to_datetime(aurora_test_mat["date"])
        aurora_test_mat = aurora_test_mat.set_index("date")
    else:
        aurora_test_mat.index = pd.to_datetime(aurora_test_mat.index)

    aurora_test_mat = aurora_test_mat.sort_index()
else:
    aurora_test_mat = pd.DataFrame()

roma_pairwise = read_table(roma_pairwise_path)
roma_decision_table = read_table(roma_decision_table_path)
roma_bootstrap_summary = read_table(roma_bootstrap_summary_path)
roma_diagnostic_summary = read_table(roma_diagnostic_summary_path)

print("ROMA strict-test matrix :", roma_test_mat.shape, roma_test_mat.index.min(), "to", roma_test_mat.index.max())
print("ROMA OOS matrix         :", roma_oos_mat.shape, roma_oos_mat.index.min(), "to", roma_oos_mat.index.max())
print("AURORA strict-test matrix:", aurora_test_mat.shape if len(aurora_test_mat) else None)
if len(aurora_test_mat):
    print("AURORA dates            :", aurora_test_mat.index.min(), "to", aurora_test_mat.index.max())

print("ROMA pairwise           :", roma_pairwise.shape)
print("ROMA decision table     :", roma_decision_table.shape)
print("ROMA bootstrap summary  :", roma_bootstrap_summary.shape)
print("ROMA diagnostic summary :", roma_diagnostic_summary.shape)

# ============================================================
# 6. Load optional AURORA Notebook 11/12 final tables
# ============================================================

print("\n" + "=" * 80)
print("Step 3: Loading optional AURORA final evidence tables")
print("=" * 80)

aurora12_table_candidates = [
    AURORA_NOTEBOOK12_ROOT / "tables" / "final_performance_table.csv",
    AURORA_NOTEBOOK12_ROOT / "tables" / "paper_final_performance_table.csv",
    AURORA_NOTEBOOK12_ROOT / "tables" / "table_106_final_performance_table.csv",
    AURORA_NOTEBOOK12_ROOT / "tables" / "final_statistical_inference_table.csv",
]

aurora11_decision_candidates = [
    AURORA_NOTEBOOK11_ROOT / "tables" / "aurora_paper_decision_table.csv",
    AURORA_NOTEBOOK11_ROOT / "tables" / "AURORA_paper_decision_table.csv",
    AURORA_NOTEBOOK11_ROOT / "tables" / "statistical_decision_table.csv",
]

available_aurora12_tables = []
for p in aurora12_table_candidates:
    if p.exists():
        try:
            tmp = read_table(p)
            available_aurora12_tables.append({
                "path": str(p),
                "shape": str(tmp.shape),
                "columns": ", ".join(map(str, tmp.columns[:20])),
            })
        except Exception as e:
            available_aurora12_tables.append({
                "path": str(p),
                "shape": "read_failed",
                "columns": repr(e),
            })

available_aurora11_tables = []
for p in aurora11_decision_candidates:
    if p.exists():
        try:
            tmp = read_table(p)
            available_aurora11_tables.append({
                "path": str(p),
                "shape": str(tmp.shape),
                "columns": ", ".join(map(str, tmp.columns[:20])),
            })
        except Exception as e:
            available_aurora11_tables.append({
                "path": str(p),
                "shape": "read_failed",
                "columns": repr(e),
            })

available_aux_tables_df = pd.DataFrame(available_aurora12_tables + available_aurora11_tables)

if len(available_aux_tables_df):
    save_table(
        available_aux_tables_df,
        local_name="notebook13_available_aurora_auxiliary_tables.csv",
        global_name=f"table_13_00_available_aurora_auxiliary_tables_{RUN_ID}.csv",
    )

print("Available auxiliary AURORA tables:")
if len(available_aux_tables_df):
    print(available_aux_tables_df.to_string(index=False))
else:
    print("None found by standard names. Continuing using loaded AURORA return matrix.")

# ============================================================
# 7. Build unified strict-test return matrix
# ============================================================

print("\n" + "=" * 80)
print("Step 4: Building unified strict-test matrix")
print("=" * 80)

# Start from AURORA matrix if available because it includes AURORA policies and many benchmarks.
unified_sources = []

if len(aurora_test_mat):
    unified_sources.append(("AURORA_N10", aurora_test_mat))

unified_sources.append(("ROMA_R3", roma_test_mat))

all_dates = None
for source_name, mat in unified_sources:
    if all_dates is None:
        all_dates = mat.index
    else:
        all_dates = all_dates.intersection(mat.index)

all_dates = pd.DatetimeIndex(all_dates).sort_values()

print("Unified common dates:", len(all_dates), all_dates.min(), "to", all_dates.max())

# Merge columns, preferring:
# - ROMA-specific policies from ROMA
# - AURORA-specific policies from AURORA
# - benchmarks from AURORA when available, otherwise ROMA
unified_mat = pd.DataFrame(index=all_dates)

if len(aurora_test_mat):
    for col in aurora_test_mat.columns:
        unified_mat[col] = aurora_test_mat.loc[all_dates, col]

for col in roma_test_mat.columns:
    if col.startswith("ROMA_"):
        unified_mat[col] = roma_test_mat.loc[all_dates, col]
    elif col not in unified_mat.columns:
        unified_mat[col] = roma_test_mat.loc[all_dates, col]

# Keep final policy order where available, plus all remaining policies.
ordered_cols = [p for p in FINAL_POLICY_ORDER if p in unified_mat.columns]
remaining_cols = [p for p in unified_mat.columns if p not in ordered_cols]
unified_mat = unified_mat[ordered_cols + remaining_cols].copy()

unified_matrix_path = RETURN_DIR / "notebook13_unified_strict_test_return_matrix.parquet"
unified_mat.to_parquet(unified_matrix_path)

unified_matrix_csv_path = RETURN_DIR / "notebook13_unified_strict_test_return_matrix.csv"
unified_mat.to_csv(unified_matrix_csv_path)

print("Unified strict-test matrix:", unified_mat.shape)
print("Columns:", unified_mat.columns.tolist())

# ============================================================
# 8. Unified performance table
# ============================================================

print("\n" + "=" * 80)
print("Step 5: Unified performance table")
print("=" * 80)

unified_perf = build_performance_table(unified_mat)

# Keep table focused for paper.
paper_policy_cols = [p for p in FINAL_POLICY_ORDER if p in unified_perf["policy_name"].values]
paper_perf = unified_perf[unified_perf["policy_name"].isin(paper_policy_cols)].copy()

paper_perf["paper_order"] = paper_perf["policy_name"].map(
    {p: i for i, p in enumerate(FINAL_POLICY_ORDER)}
).fillna(999).astype(int)

paper_perf = paper_perf.sort_values(
    ["composite_rank", "paper_order"],
    ascending=[True, True],
).reset_index(drop=True)

# Add interpretation labels.
paper_perf["method_role"] = paper_perf["policy_name"].map({
    PRIMARY_AURORA_POLICY: "Final AURORA method",
    "AURORA10_UAMV_D_low_turnover": "AURORA sensitivity",
    "AURORA10_UAMV_E_no_regime_tilt_control": "AURORA sensitivity",
    "AURORA10_UAMV_A_balanced": "AURORA sensitivity",
    "AURORA10_UAMV_C_more60_growth": "AURORA sensitivity",
    VALIDATION_SELECTED_AURORA_POLICY: "AURORA validation-selected comparator",
    PRIMARY_ROMA_POLICY: "ROMA regime-template baseline",
    "ROMA_P2_20d_only_return_seeking": "ROMA regime-template baseline",
    "ROMA_P0_validation_selected_20_60_blend": "ROMA regime-template baseline",
    "B12_ma_timing_equal_weight": "Strong timing benchmark",
    "B6_00881_only": "Strong single-ETF benchmark",
    "B3_0050_only": "Market ETF benchmark",
    "B1_equal_weight_all_etfs": "Passive diversified benchmark",
    "B10_momentum_top2_63d": "Momentum benchmark",
    "B16_minimum_variance_126d": "Risk-based benchmark",
    "B15_minimum_variance_126d": "Risk-based benchmark",
}).fillna("Other")

save_table(
    unified_perf,
    local_name="notebook13_unified_all_policy_performance.csv",
    global_name=f"table_13_01_unified_all_policy_performance_{RUN_ID}.csv",
)

save_table(
    paper_perf,
    local_name="notebook13_paper_focused_performance_table.csv",
    global_name=f"table_13_02_paper_focused_performance_table_{RUN_ID}.csv",
)

print("Paper-focused performance table:")
print(
    paper_perf[
        [
            "display_name",
            "method_role",
            "policy_group",
            "n_days",
            "total_return",
            "annual_return",
            "annual_volatility",
            "sharpe",
            "sortino",
            "max_drawdown",
            "calmar",
            "composite_rank",
        ]
    ].to_string(index=False)
)

# ============================================================
# 9. Pairwise evidence: ROMA vs AURORA vs benchmarks
# ============================================================

print("\n" + "=" * 80)
print("Step 6: Pairwise evidence table")
print("=" * 80)

pairwise_pairs = [
    (PRIMARY_AURORA_POLICY, "B12_ma_timing_equal_weight", "AURORA vs best timing benchmark"),
    (PRIMARY_AURORA_POLICY, "B6_00881_only", "AURORA vs high-return ETF benchmark"),
    (PRIMARY_AURORA_POLICY, "B3_0050_only", "AURORA vs market ETF benchmark"),
    (PRIMARY_AURORA_POLICY, "B1_equal_weight_all_etfs", "AURORA vs passive diversified benchmark"),
    (PRIMARY_AURORA_POLICY, PRIMARY_ROMA_POLICY, "AURORA vs ROMA baseline"),
    (PRIMARY_ROMA_POLICY, "B12_ma_timing_equal_weight", "ROMA vs best timing benchmark"),
    (PRIMARY_ROMA_POLICY, "B6_00881_only", "ROMA vs high-return ETF benchmark"),
    (PRIMARY_ROMA_POLICY, "B3_0050_only", "ROMA vs market ETF benchmark"),
    (PRIMARY_ROMA_POLICY, "B1_equal_weight_all_etfs", "ROMA vs passive diversified benchmark"),
    ("ROMA_P2_20d_only_return_seeking", PRIMARY_AURORA_POLICY, "ROMA 20d vs AURORA"),
]

pairwise_rows = []

for strategy, benchmark, comparison_label in pairwise_pairs:
    if strategy not in unified_mat.columns or benchmark not in unified_mat.columns:
        continue

    m = paired_difference_metrics(unified_mat[strategy], unified_mat[benchmark])
    pairwise_rows.append({
        "comparison_label": comparison_label,
        "strategy_policy": strategy,
        "strategy_display_name": POLICY_DISPLAY_NAMES.get(strategy, strategy),
        "benchmark_policy": benchmark,
        "benchmark_display_name": POLICY_DISPLAY_NAMES.get(benchmark, benchmark),
        **m,
    })

unified_pairwise_df = pd.DataFrame(pairwise_rows)

save_table(
    unified_pairwise_df,
    local_name="notebook13_unified_pairwise_observed_differences.csv",
    global_name=f"table_13_03_unified_pairwise_observed_differences_{RUN_ID}.csv",
)

print("Unified pairwise observed differences:")
print(
    unified_pairwise_df[
        [
            "comparison_label",
            "strategy_display_name",
            "benchmark_display_name",
            "n_days",
            "diff_total_return",
            "diff_sharpe",
            "diff_sortino",
            "drawdown_improvement",
            "diff_calmar",
            "annualized_mean_excess_return",
        ]
    ].to_string(index=False)
)

# ============================================================
# 10. Statistical evidence summary
# ============================================================

print("\n" + "=" * 80)
print("Step 7: Statistical evidence summary")
print("=" * 80)

# R3 decision table was constructed as strategy - comparator.
# For ROMA vs AURORA, it contains ROMA - AURORA.
# For final paper, we also construct an AURORA-centric interpretation by flipping signs.
decision = roma_decision_table.copy()

# Normalize bool columns if loaded as strings.
for c in ["significant_positive_95", "significant_negative_95", "not_significant_95"]:
    if c in decision.columns:
        if decision[c].dtype == object:
            decision[c] = decision[c].astype(str).str.lower().map({
                "true": True,
                "false": False,
            }).fillna(False)

# Core rows from R3.
core_decision_rows = []

def add_decision_rows_from_r3(strategy, benchmark, label, flip=False):
    rows = decision[
        (decision["strategy_policy"] == strategy)
        & (decision["benchmark_policy"] == benchmark)
    ].copy()

    if rows.empty:
        return

    for _, r in rows.iterrows():
        observed = float(r["observed"])
        ci2 = float(r["ci_2p5"])
        ci97 = float(r["ci_97p5"])

        if flip:
            observed_final = -observed
            ci2_final = -ci97
            ci97_final = -ci2

            if bool(r.get("significant_positive_95", False)):
                decision_final = "significantly_negative_at_95pct_bootstrap"
            elif bool(r.get("significant_negative_95", False)):
                decision_final = "significantly_positive_at_95pct_bootstrap"
            else:
                decision_final = "not_significant_at_95pct_bootstrap"
        else:
            observed_final = observed
            ci2_final = ci2
            ci97_final = ci97
            decision_final = r.get("decision", "unknown")

        metric = r["metric"]

        if metric == "drawdown_improvement":
            metric_meaning = "Positive means strategy has less severe maximum drawdown."
        elif metric == "diff_total_return":
            metric_meaning = "Positive means strategy has higher cumulative return."
        elif metric == "diff_sharpe":
            metric_meaning = "Positive means strategy has higher Sharpe ratio."
        elif metric == "diff_sortino":
            metric_meaning = "Positive means strategy has higher Sortino ratio."
        elif metric == "diff_calmar":
            metric_meaning = "Positive means strategy has higher Calmar ratio."
        elif metric == "annualized_mean_excess_return":
            metric_meaning = "Positive means strategy has higher mean daily return annualized."
        else:
            metric_meaning = ""

        core_decision_rows.append({
            "comparison_label": label,
            "strategy_policy": benchmark if flip else strategy,
            "strategy_display_name": POLICY_DISPLAY_NAMES.get(benchmark if flip else strategy, benchmark if flip else strategy),
            "comparator_policy": strategy if flip else benchmark,
            "comparator_display_name": POLICY_DISPLAY_NAMES.get(strategy if flip else benchmark, strategy if flip else benchmark),
            "metric": metric,
            "observed": observed_final,
            "ci_2p5": ci2_final,
            "ci_97p5": ci97_final,
            "decision": decision_final,
            "metric_meaning": metric_meaning,
            "source": "ROMA_R3_block_bootstrap",
        })

# ROMA-centric from R3.
add_decision_rows_from_r3(
    PRIMARY_ROMA_POLICY,
    "B12_ma_timing_equal_weight",
    "ROMA vs B12 timing benchmark",
    flip=False,
)

add_decision_rows_from_r3(
    PRIMARY_ROMA_POLICY,
    PRIMARY_AURORA_POLICY,
    "ROMA vs AURORA10-UAMV-B",
    flip=False,
)

# AURORA-centric view by flipping ROMA vs AURORA.
add_decision_rows_from_r3(
    PRIMARY_ROMA_POLICY,
    PRIMARY_AURORA_POLICY,
    "AURORA10-UAMV-B vs ROMA",
    flip=True,
)

statistical_evidence_df = pd.DataFrame(core_decision_rows)

save_table(
    statistical_evidence_df,
    local_name="notebook13_statistical_evidence_summary.csv",
    global_name=f"table_13_04_statistical_evidence_summary_{RUN_ID}.csv",
)

print("Statistical evidence summary:")
if len(statistical_evidence_df):
    print(
        statistical_evidence_df[
            [
                "comparison_label",
                "strategy_display_name",
                "comparator_display_name",
                "metric",
                "observed",
                "ci_2p5",
                "ci_97p5",
                "decision",
            ]
        ].to_string(index=False)
    )
else:
    print("No statistical evidence rows found.")

# ============================================================
# 11. Methodology progression table
# ============================================================

print("\n" + "=" * 80)
print("Step 8: Methodology progression table")
print("=" * 80)

progression_rows = [
    {
        "stage": "ROMA R1",
        "method": "Purged walk-forward ROMA regime probability rebuild",
        "purpose": "Create leakage-controlled regime probabilities using the shared AURORA feature-label dataset.",
        "selection_rule": "Validation aggregate composite rank.",
        "main_finding": "20-day regime signal was more usable than the weak 60-day signal.",
        "paper_role": "Forecasting baseline construction.",
    },
    {
        "stage": "ROMA R2",
        "method": "Pre-defined regime-template allocation",
        "purpose": "Convert ROMA probabilities into ETF weights using interpretable return-seeking templates.",
        "selection_rule": "Templates pre-defined; no strict-test optimization.",
        "main_finding": "Best ROMA policy underperformed B12 moving-average timing on strict-test return, Sharpe, drawdown, and composite rank.",
        "paper_role": "Negative baseline / ablation.",
    },
    {
        "stage": "ROMA R3",
        "method": "Paired circular block bootstrap inference",
        "purpose": "Test whether ROMA outperformed benchmarks or AURORA.",
        "selection_rule": "No new selection; tests R2-generated returns.",
        "main_finding": "ROMA showed no statistically significant advantage over B12 and was risk-adjusted weaker than AURORA10-UAMV-B.",
        "paper_role": "Statistical evidence for insufficiency of simple regime-template mapping.",
    },
    {
        "stage": "AURORA Notebook 10",
        "method": "Uncertainty-aware mean-variance allocation",
        "purpose": "Use regime probabilities, uncertainty, bearish risk, shrinkage, and constraints to construct risk-aware ETF weights.",
        "selection_rule": "AURORA10-UAMV-B selected as final risk-control method in the final analysis.",
        "main_finding": "AURORA10-UAMV-B improved risk-adjusted behavior and substantially reduced drawdown relative to high-return benchmarks, while sacrificing total return.",
        "paper_role": "Final proposed allocation method.",
    },
    {
        "stage": "AURORA Notebook 11",
        "method": "Block bootstrap statistical inference",
        "purpose": "Test AURORA10-UAMV-B against primary benchmarks.",
        "selection_rule": "No new selection within inference notebook.",
        "main_finding": "The strongest supported claim is significant maximum-drawdown reduction, not total-return or Sharpe superiority.",
        "paper_role": "Main statistical support.",
    },
    {
        "stage": "Notebook 13",
        "method": "Unified ROMA-AURORA comparison",
        "purpose": "Combine ROMA baseline evidence, AURORA evidence, and benchmark comparisons into final paper tables.",
        "selection_rule": "Report-only synthesis.",
        "main_finding": "Regime forecasting alone was insufficient; the allocation layer is central to robust downside-risk management.",
        "paper_role": "Final manuscript evidence synthesis.",
    },
]

progression_df = pd.DataFrame(progression_rows)

save_table(
    progression_df,
    local_name="notebook13_methodology_progression.csv",
    global_name=f"table_13_05_methodology_progression_{RUN_ID}.csv",
)

print("Methodology progression:")
print(progression_df.to_string(index=False))

# ============================================================
# 12. Claim checklist
# ============================================================

print("\n" + "=" * 80)
print("Step 9: Claim checklist")
print("=" * 80)

def evidence_decision_for(comparison_label, metric):
    if statistical_evidence_df.empty:
        return None
    rows = statistical_evidence_df[
        (statistical_evidence_df["comparison_label"] == comparison_label)
        & (statistical_evidence_df["metric"] == metric)
    ]
    if rows.empty:
        return None
    return rows.iloc[0].to_dict()

claim_rows = []

# Claim 1: ROMA does not beat B12.
roma_b12_total = evidence_decision_for("ROMA vs B12 timing benchmark", "diff_total_return")
roma_b12_sharpe = evidence_decision_for("ROMA vs B12 timing benchmark", "diff_sharpe")
roma_b12_dd = evidence_decision_for("ROMA vs B12 timing benchmark", "drawdown_improvement")

claim_rows.append({
    "claim": "ROMA regime-template allocation does not outperform the B12 moving-average timing benchmark.",
    "supported": True,
    "evidence": (
        "Observed ROMA-P4 underperformed B12 in total return, Sharpe, Sortino, drawdown, and Calmar; "
        "bootstrap results show no significant ROMA advantage."
    ),
    "safe_paper_wording": (
        "Under aligned strict-test evaluation, the ROMA regime-template baseline did not outperform the "
        "moving-average timing benchmark."
    ),
    "avoid_wording": "ROMA is a superior allocation method.",
})

# Claim 2: AURORA beats ROMA on risk-adjusted/downside.
aurora_roma_sharpe = evidence_decision_for("AURORA10-UAMV-B vs ROMA", "diff_sharpe")
aurora_roma_sortino = evidence_decision_for("AURORA10-UAMV-B vs ROMA", "diff_sortino")
aurora_roma_dd = evidence_decision_for("AURORA10-UAMV-B vs ROMA", "drawdown_improvement")

claim_rows.append({
    "claim": "AURORA10-UAMV-B provides stronger risk-adjusted and downside-risk behavior than ROMA.",
    "supported": True,
    "evidence": (
        "R3 ROMA-vs-AURORA tests show ROMA was significantly worse than AURORA10-UAMV-B in Sharpe, "
        "Sortino, and maximum-drawdown improvement; equivalently, AURORA is significantly better "
        "on these risk metrics."
    ),
    "safe_paper_wording": (
        "Compared with the ROMA regime-template baseline, AURORA10-UAMV-B achieved significantly "
        "stronger risk-adjusted and downside-risk behavior in the strict-test period."
    ),
    "avoid_wording": "AURORA guarantees better investment performance.",
})

# Claim 3: AURORA is not total-return dominant.
claim_rows.append({
    "claim": "AURORA is not total-return dominant.",
    "supported": True,
    "evidence": (
        "AURORA10-UAMV-B sacrifices total return relative to high-return ETF benchmarks and ROMA in some observed comparisons; "
        "its strongest supported contribution is downside-risk control."
    ),
    "safe_paper_wording": (
        "AURORA10-UAMV-B should be interpreted as a downside-risk control layer rather than a return-maximizing strategy."
    ),
    "avoid_wording": "AURORA maximizes returns or beats all benchmarks in total return.",
})

# Claim 4: Allocation layer matters.
claim_rows.append({
    "claim": "The allocation layer is critical; regime forecasting alone is insufficient.",
    "supported": True,
    "evidence": (
        "ROMA uses regime probabilities with simple templates and underperforms strong benchmarks, while AURORA's "
        "uncertainty-aware allocation layer produces stronger downside-risk behavior."
    ),
    "safe_paper_wording": (
        "The results indicate that regime forecasts require a carefully designed uncertainty-aware allocation layer "
        "to translate signals into robust portfolio behavior."
    ),
    "avoid_wording": "Better forecasting accuracy alone explains the portfolio results.",
})

claim_checklist_df = pd.DataFrame(claim_rows)

save_table(
    claim_checklist_df,
    local_name="notebook13_claim_checklist.csv",
    global_name=f"table_13_06_claim_checklist_{RUN_ID}.csv",
)

print("Claim checklist:")
print(claim_checklist_df.to_string(index=False))

# ============================================================
# 13. Figures
# ============================================================

print("\n" + "=" * 80)
print("Step 10: Creating paper figures")
print("=" * 80)

figure_records = []

def select_plot_policies():
    candidates = [
        PRIMARY_AURORA_POLICY,
        PRIMARY_ROMA_POLICY,
        "B12_ma_timing_equal_weight",
        "B6_00881_only",
        "B3_0050_only",
        "B1_equal_weight_all_etfs",
        "B10_momentum_top2_63d",
    ]
    return [p for p in candidates if p in unified_mat.columns]

PLOT_POLICIES = select_plot_policies()

def save_figure_record(path, figure_id, title, caption):
    figure_records.append({
        "figure_id": figure_id,
        "path": str(path),
        "title": title,
        "caption": caption,
    })

# Figure 1: Equity curves.
plt.figure(figsize=(12, 6))
for policy in PLOT_POLICIES:
    equity = make_equity(unified_mat[policy])
    plt.plot(equity.index, equity.values, linewidth=1.8, label=POLICY_DISPLAY_NAMES.get(policy, policy))

plt.title("Unified strict-test equity curves")
plt.xlabel("Date")
plt.ylabel("Equity")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
fig1_path = PAPER_FIGURE_DIR / "figure13_01_unified_equity_curves.png"
plt.savefig(fig1_path, dpi=220)
plt.close()

save_figure_record(
    fig1_path,
    "Figure 13.1",
    "Unified strict-test equity curves",
    "Equity curves for AURORA10-UAMV-B, ROMA-P4, and primary benchmarks on the aligned 319-day strict-test period.",
)

# Figure 2: Drawdown curves.
plt.figure(figsize=(12, 6))
for policy in PLOT_POLICIES:
    dd = make_drawdown(unified_mat[policy])
    plt.plot(dd.index, dd.values, linewidth=1.8, label=POLICY_DISPLAY_NAMES.get(policy, policy))

plt.title("Unified strict-test drawdowns")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
fig2_path = PAPER_FIGURE_DIR / "figure13_02_unified_drawdowns.png"
plt.savefig(fig2_path, dpi=220)
plt.close()

save_figure_record(
    fig2_path,
    "Figure 13.2",
    "Unified strict-test drawdowns",
    "Drawdown curves showing the downside-risk behavior of AURORA10-UAMV-B, ROMA-P4, and primary benchmarks.",
)

# Figure 3: Sharpe bar.
plot_perf = paper_perf[paper_perf["policy_name"].isin(PLOT_POLICIES)].copy()
plot_perf = plot_perf.sort_values("sharpe", ascending=True)

colors = []
for p in plot_perf["policy_name"]:
    if p == PRIMARY_AURORA_POLICY:
        colors.append("#1f77b4")
    elif p == PRIMARY_ROMA_POLICY:
        colors.append("#d62728")
    else:
        colors.append("#7f7f7f")

plt.figure(figsize=(10, 6))
plt.barh(plot_perf["display_name"], plot_perf["sharpe"], color=colors)
plt.xlabel("Sharpe ratio")
plt.title("Strict-test Sharpe comparison")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
fig3_path = PAPER_FIGURE_DIR / "figure13_03_strict_test_sharpe_bar.png"
plt.savefig(fig3_path, dpi=220)
plt.close()

save_figure_record(
    fig3_path,
    "Figure 13.3",
    "Strict-test Sharpe comparison",
    "Sharpe ratios for AURORA, ROMA, and primary benchmarks on the aligned strict-test period.",
)

# Figure 4: Max drawdown bar.
plot_perf = paper_perf[paper_perf["policy_name"].isin(PLOT_POLICIES)].copy()
plot_perf = plot_perf.sort_values("max_drawdown", ascending=True)

colors = []
for p in plot_perf["policy_name"]:
    if p == PRIMARY_AURORA_POLICY:
        colors.append("#1f77b4")
    elif p == PRIMARY_ROMA_POLICY:
        colors.append("#d62728")
    else:
        colors.append("#7f7f7f")

plt.figure(figsize=(10, 6))
plt.barh(plot_perf["display_name"], plot_perf["max_drawdown"], color=colors)
plt.xlabel("Maximum drawdown")
plt.title("Strict-test maximum drawdown comparison")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
fig4_path = PAPER_FIGURE_DIR / "figure13_04_strict_test_max_drawdown_bar.png"
plt.savefig(fig4_path, dpi=220)
plt.close()

save_figure_record(
    fig4_path,
    "Figure 13.4",
    "Strict-test maximum drawdown comparison",
    "Maximum drawdown comparison. Less negative values indicate smaller drawdowns.",
)

# Figure 5: Return-risk scatter.
plot_perf = paper_perf[paper_perf["policy_name"].isin(PLOT_POLICIES)].copy()

plt.figure(figsize=(8, 6))
for _, row in plot_perf.iterrows():
    p = row["policy_name"]
    if p == PRIMARY_AURORA_POLICY:
        color = "#1f77b4"
        size = 120
    elif p == PRIMARY_ROMA_POLICY:
        color = "#d62728"
        size = 120
    else:
        color = "#7f7f7f"
        size = 70

    plt.scatter(
        row["annual_volatility"],
        row["annual_return"],
        s=size,
        color=color,
        alpha=0.85,
    )
    plt.text(
        row["annual_volatility"],
        row["annual_return"],
        POLICY_DISPLAY_NAMES.get(p, p),
        fontsize=8,
        ha="left",
        va="bottom",
    )

plt.xlabel("Annualized volatility")
plt.ylabel("Annualized return")
plt.title("Strict-test return-risk trade-off")
plt.grid(True, alpha=0.3)
plt.tight_layout()
fig5_path = PAPER_FIGURE_DIR / "figure13_05_return_risk_scatter.png"
plt.savefig(fig5_path, dpi=220)
plt.close()

save_figure_record(
    fig5_path,
    "Figure 13.5",
    "Strict-test return-risk trade-off",
    "Annualized return versus annualized volatility for AURORA, ROMA, and primary benchmarks.",
)

# Figure 6: Statistical evidence heatmap.
if len(statistical_evidence_df):
    heat = statistical_evidence_df[
        statistical_evidence_df["metric"].isin([
            "diff_total_return",
            "diff_sharpe",
            "diff_sortino",
            "drawdown_improvement",
            "diff_calmar",
            "annualized_mean_excess_return",
        ])
    ].copy()

    heat["row_label"] = heat["comparison_label"]
    heat_pivot = heat.pivot_table(
        index="row_label",
        columns="metric",
        values="observed",
        aggfunc="first",
    )

    plt.figure(figsize=(12, max(4, 0.6 * len(heat_pivot))))
    if HAS_SEABORN:
        sns.heatmap(
            heat_pivot,
            center=0,
            cmap="RdBu",
            annot=True,
            fmt=".3f",
            linewidths=0.5,
        )
    else:
        plt.imshow(heat_pivot.values, aspect="auto")
        plt.colorbar()
        plt.xticks(range(len(heat_pivot.columns)), heat_pivot.columns, rotation=45, ha="right")
        plt.yticks(range(len(heat_pivot.index)), heat_pivot.index)

    plt.title("Observed pairwise differences from statistical evidence table")
    plt.tight_layout()
    fig6_path = PAPER_FIGURE_DIR / "figure13_06_statistical_evidence_heatmap.png"
    plt.savefig(fig6_path, dpi=220)
    plt.close()

    save_figure_record(
        fig6_path,
        "Figure 13.6",
        "Observed pairwise statistical evidence",
        "Observed pairwise differences for selected comparisons. Positive values favor the strategy named in each row.",
    )

figure_index_df = pd.DataFrame(figure_records)

save_table(
    figure_index_df,
    local_name="notebook13_figure_index.csv",
    global_name=f"table_13_07_figure_index_{RUN_ID}.csv",
)

print("Figure index:")
print(figure_index_df.to_string(index=False))

# ============================================================
# 14. Paper-ready summary tables
# ============================================================

print("\n" + "=" * 80)
print("Step 11: Paper-ready summary tables")
print("=" * 80)

# Compact final performance table.
compact_perf_cols = [
    "display_name",
    "method_role",
    "policy_group",
    "n_days",
    "total_return",
    "annual_return",
    "annual_volatility",
    "sharpe",
    "sortino",
    "max_drawdown",
    "calmar",
    "composite_rank",
]

compact_final_perf = paper_perf[compact_perf_cols].copy()

# Round for manuscript.
rounded_perf = compact_final_perf.copy()
for c in [
    "total_return",
    "annual_return",
    "annual_volatility",
    "sharpe",
    "sortino",
    "max_drawdown",
    "calmar",
    "composite_rank",
]:
    if c in rounded_perf.columns:
        rounded_perf[c] = rounded_perf[c].astype(float).round(4)

save_table(
    compact_final_perf,
    local_name="notebook13_final_compact_performance_table.csv",
    global_name=f"table_13_08_final_compact_performance_table_{RUN_ID}.csv",
)

save_table(
    rounded_perf,
    local_name="notebook13_final_compact_performance_table_rounded.csv",
    global_name=f"table_13_09_final_compact_performance_table_rounded_{RUN_ID}.csv",
)

# Compact statistical evidence table.
if len(statistical_evidence_df):
    compact_stat = statistical_evidence_df[
        [
            "comparison_label",
            "strategy_display_name",
            "comparator_display_name",
            "metric",
            "observed",
            "ci_2p5",
            "ci_97p5",
            "decision",
            "metric_meaning",
        ]
    ].copy()

    rounded_stat = compact_stat.copy()
    for c in ["observed", "ci_2p5", "ci_97p5"]:
        rounded_stat[c] = rounded_stat[c].astype(float).round(4)

    save_table(
        compact_stat,
        local_name="notebook13_final_statistical_evidence_table.csv",
        global_name=f"table_13_10_final_statistical_evidence_table_{RUN_ID}.csv",
    )

    save_table(
        rounded_stat,
        local_name="notebook13_final_statistical_evidence_table_rounded.csv",
        global_name=f"table_13_11_final_statistical_evidence_table_rounded_{RUN_ID}.csv",
    )
else:
    compact_stat = pd.DataFrame()
    rounded_stat = pd.DataFrame()

print("Rounded performance table:")
print(rounded_perf.to_string(index=False))

print("\nRounded statistical table:")
if len(rounded_stat):
    print(rounded_stat.to_string(index=False))
else:
    print("No statistical table.")

# ============================================================
# 15. Manuscript text assets
# ============================================================

print("\n" + "=" * 80)
print("Step 12: Writing manuscript text assets")
print("=" * 80)

# Extract key numbers.
def get_perf(policy):
    rows = paper_perf[paper_perf["policy_name"] == policy]
    if rows.empty:
        return None
    return rows.iloc[0].to_dict()

aurora_perf = get_perf(PRIMARY_AURORA_POLICY)
roma_perf = get_perf(PRIMARY_ROMA_POLICY)
b12_perf = get_perf("B12_ma_timing_equal_weight")
b6_perf = get_perf("B6_00881_only")
b3_perf = get_perf("B3_0050_only")
b1_perf = get_perf("B1_equal_weight_all_etfs")

def fmt(x, digits=4):
    if x is None or pd.isna(x):
        return "NA"
    return f"{float(x):.{digits}f}"

methods_summary = f"""
## Methods summary for final manuscript

We evaluate a unified Taiwan ETF allocation framework using an aligned strict-test period of {len(unified_mat)} trading days from {unified_mat.index.min().date()} to {unified_mat.index.max().date()}. The ETF universe consists of Taiwan ETFs 0050, 006208, 00692, and 00881, with cash allowed as a defensive allocation.

The ROMA branch is reconstructed as a leakage-controlled regime-template baseline. ROMA first generates purged walk-forward regime probabilities, then maps regime probabilities into pre-defined ETF allocation templates. No ROMA template is optimized on the strict-test period.

The AURORA branch uses the same leakage-controlled forecasting foundation but replaces the simple regime-to-template mapping with an uncertainty-aware mean-variance allocation layer. The final AURORA policy, AURORA10-UAMV-B, emphasizes the 60-day regime horizon, increases risk aversion under uncertainty and bearish probability, uses shrinkage in return estimation, applies covariance-based risk control, and enforces portfolio constraints.

All ROMA, AURORA, and benchmark policies are compared on identical strict-test dates. Statistical evidence is summarized using paired circular block bootstrap inference, with positive drawdown improvement indicating a less severe maximum drawdown for the strategy relative to the comparator.
""".strip()

results_summary = f"""
## Results summary for final manuscript

On the aligned strict-test period, the strongest ROMA regime-template policy was {POLICY_DISPLAY_NAMES.get(PRIMARY_ROMA_POLICY, PRIMARY_ROMA_POLICY)}, with total return {fmt(roma_perf['total_return'])}, Sharpe ratio {fmt(roma_perf['sharpe'])}, and maximum drawdown {fmt(roma_perf['max_drawdown'])}. This was weaker than the moving-average timing benchmark {POLICY_DISPLAY_NAMES.get('B12_ma_timing_equal_weight')}, which achieved total return {fmt(b12_perf['total_return'])}, Sharpe ratio {fmt(b12_perf['sharpe'])}, and maximum drawdown {fmt(b12_perf['max_drawdown'])}.

The final AURORA policy, {POLICY_DISPLAY_NAMES.get(PRIMARY_AURORA_POLICY, PRIMARY_AURORA_POLICY)}, achieved total return {fmt(aurora_perf['total_return'])}, Sharpe ratio {fmt(aurora_perf['sharpe'])}, and maximum drawdown {fmt(aurora_perf['max_drawdown'])}. Relative to ROMA, AURORA provided stronger risk-adjusted and downside-risk behavior. Relative to high-return ETF benchmarks, AURORA sacrificed total return but substantially improved downside-risk control.

The final evidence supports a risk-control interpretation rather than a return-dominance interpretation. Simple regime-template allocation was insufficient to outperform strong timing and passive benchmarks. The stronger contribution emerged from the uncertainty-aware allocation layer, which converted noisy regime probabilities into a more defensive, risk-controlled ETF allocation.
""".strip()

discussion_summary = """
## Discussion summary for final manuscript

The unified ROMA-AURORA comparison indicates that regime forecasts alone are not sufficient for robust ETF allocation. ROMA provides an interpretable and leakage-controlled regime-template baseline, but its strict-test performance remains weaker than simple technical timing and passive benchmarks. This finding is important because it prevents the study from overstating the value of regime classification by itself.

AURORA improves the allocation stage by incorporating uncertainty, bearish probability, shrinkage, risk aversion, and portfolio constraints. The empirical results suggest that the allocation layer is central to transforming probabilistic regime signals into robust portfolio behavior. The strongest supported contribution is downside-risk management, especially maximum-drawdown reduction, rather than statistically proven total-return superiority.

The main limitations are the short strict-test period, the small ETF universe, the Taiwan-specific setting, and the fact that the final AURORA variant should be interpreted as a research result rather than a guaranteed investment rule. Future work should extend the ETF universe, test additional market regimes, and pre-register allocation variants before final holdout evaluation.
""".strip()

abstract_draft = f"""
## Draft abstract

Regime forecasts are often used to guide tactical asset allocation, but the link between forecasting signals and robust portfolio construction remains fragile. We study this issue in Taiwan ETF allocation using a leakage-controlled experimental design. First, we reconstruct ROMA, a regime-template allocation baseline that maps purged walk-forward regime probabilities into pre-defined ETF weights. Second, we compare it with AURORA, an uncertainty-aware allocation framework that incorporates probabilistic regime forecasts, uncertainty, bearish-risk signals, shrinkage, and constrained mean-variance optimization. On an aligned strict-test period of {len(unified_mat)} trading days, the best ROMA policy underperformed a moving-average timing benchmark and did not provide statistically supported superiority. In contrast, AURORA10-UAMV-B produced stronger risk-adjusted and downside-risk behavior, although it did not establish total-return dominance. The results show that regime forecasting alone is insufficient for robust ETF allocation; the allocation layer is critical for translating uncertain forecasts into downside-risk control.
""".strip()

conclusion_draft = """
## Draft conclusion

This study compared regime-template allocation and uncertainty-aware allocation under a common leakage-controlled protocol for Taiwan ETF portfolios. The reconstructed ROMA baseline showed that simple regime-to-weight templates were not sufficient to outperform strong timing and passive benchmarks. This negative result is informative: it demonstrates that regime labels or probabilities do not automatically translate into robust portfolio performance.

The final AURORA allocation layer improved the use of probabilistic regime signals by incorporating uncertainty-aware risk control, bearish-risk penalties, shrinkage, and portfolio constraints. The strongest supported empirical contribution is downside-risk management rather than return maximization. AURORA10-UAMV-B should therefore be interpreted as a forecast-conditioned risk-control allocation method, not as a guaranteed alpha strategy.

Overall, the unified ROMA-AURORA evidence suggests that future forecasting-based allocation systems should be evaluated not only by predictive accuracy or total return, but also by how uncertainty is transformed into portfolio-level risk decisions.
""".strip()

limitations_text = """
## Limitations

1. The strict-test period contains 319 trading days, which is useful for aligned comparison but short for broad claims about long-run market performance.
2. The ETF universe is limited to four Taiwan ETFs and cash, so the findings may not generalize to larger or international universes.
3. The strongest AURORA evidence concerns drawdown and risk-adjusted behavior; total-return dominance is not statistically established.
4. ROMA and AURORA are research prototypes and should not be interpreted as personalized financial advice or investment recommendations.
5. Model and allocation variants should ideally be pre-specified before a future untouched holdout evaluation.
""".strip()

figure_captions_text = "\n\n".join([
    f"**{row['figure_id']}. {row['title']}.** {row['caption']}"
    for _, row in figure_index_df.iterrows()
])

claim_boundary_text = """
## Claim boundary

Supported claim:
AURORA-TWETF improves downside-risk control and risk-adjusted behavior relative to a reconstructed ROMA regime-template baseline, and the evidence indicates that regime forecasting alone is insufficient for robust ETF allocation.

Not supported:
The experiments do not prove total-return dominance, guaranteed future outperformance, or personalized investment suitability.
""".strip()

combined_manuscript_assets = "\n\n".join([
    abstract_draft,
    methods_summary,
    results_summary,
    discussion_summary,
    conclusion_draft,
    limitations_text,
    claim_boundary_text,
    "## Figure captions",
    figure_captions_text,
])

text_assets = {
    "methods_summary.md": methods_summary,
    "results_summary.md": results_summary,
    "discussion_summary.md": discussion_summary,
    "abstract_draft.md": abstract_draft,
    "conclusion_draft.md": conclusion_draft,
    "limitations.md": limitations_text,
    "claim_boundary.md": claim_boundary_text,
    "figure_captions.md": figure_captions_text,
    "combined_manuscript_assets.md": combined_manuscript_assets,
}

for filename, text in text_assets.items():
    save_markdown(MANUSCRIPT_RUN_DIR / filename, text)
    save_markdown(MANUSCRIPT_DIR / f"{RUN_ID}_{filename}", text)

print("Manuscript assets saved to:", MANUSCRIPT_RUN_DIR)

# ============================================================
# 16. Output index
# ============================================================

print("\n" + "=" * 80)
print("Step 13: Creating final output index")
print("=" * 80)

output_index_rows = [
    {
        "artifact_type": "return_matrix",
        "name": "unified_strict_test_return_matrix",
        "path": str(unified_matrix_path),
        "description": "Unified strict-test daily return matrix for ROMA, AURORA, and benchmarks.",
    },
    {
        "artifact_type": "table",
        "name": "unified_all_policy_performance",
        "path": str(TABLE_RUN_DIR / "notebook13_unified_all_policy_performance.csv"),
        "description": "All available policies ranked by performance metrics.",
    },
    {
        "artifact_type": "table",
        "name": "paper_focused_performance_table",
        "path": str(TABLE_RUN_DIR / "notebook13_paper_focused_performance_table.csv"),
        "description": "Paper-focused strict-test performance table.",
    },
    {
        "artifact_type": "table",
        "name": "final_compact_performance_table_rounded",
        "path": str(TABLE_RUN_DIR / "notebook13_final_compact_performance_table_rounded.csv"),
        "description": "Rounded performance table for manuscript.",
    },
    {
        "artifact_type": "table",
        "name": "statistical_evidence_summary",
        "path": str(TABLE_RUN_DIR / "notebook13_statistical_evidence_summary.csv"),
        "description": "Bootstrap-based statistical evidence summary.",
    },
    {
        "artifact_type": "table",
        "name": "claim_checklist",
        "path": str(TABLE_RUN_DIR / "notebook13_claim_checklist.csv"),
        "description": "Supported and unsupported manuscript claims.",
    },
    {
        "artifact_type": "table",
        "name": "methodology_progression",
        "path": str(TABLE_RUN_DIR / "notebook13_methodology_progression.csv"),
        "description": "Methodological progression from ROMA to AURORA.",
    },
    {
        "artifact_type": "table",
        "name": "figure_index",
        "path": str(TABLE_RUN_DIR / "notebook13_figure_index.csv"),
        "description": "Index of paper-ready figures.",
    },
    {
        "artifact_type": "manuscript",
        "name": "combined_manuscript_assets",
        "path": str(MANUSCRIPT_RUN_DIR / "combined_manuscript_assets.md"),
        "description": "Draft abstract, methods, results, discussion, conclusion, limitations, and captions.",
    },
]

for _, row in figure_index_df.iterrows():
    output_index_rows.append({
        "artifact_type": "figure",
        "name": row["figure_id"],
        "path": row["path"],
        "description": row["caption"],
    })

output_index_df = pd.DataFrame(output_index_rows)

save_table(
    output_index_df,
    local_name="notebook13_output_index.csv",
    global_name=f"table_13_12_output_index_{RUN_ID}.csv",
)

print("Output index:")
print(output_index_df.to_string(index=False))

# ============================================================
# 17. Validation report and manifest
# ============================================================

print("\n" + "=" * 80)
print("Step 14: Saving validation report and SHA256 manifest")
print("=" * 80)

validation_report = {
    "project_code": PROJECT_CODE,
    "notebook": "13_ROMA_AURORA_unified_comparison.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "purpose": (
        "Final unified ROMA-AURORA comparison for paper tables, figures, and manuscript assets."
    ),
    "input_paths": {
        "notebook13_input_index": str(NOTEBOOK13_INPUT_INDEX_PATH),
        "roma_test_matrix": str(roma_test_matrix_path),
        "aurora_test_matrix": str(aurora_test_matrix_path) if aurora_test_matrix_path is not None else None,
        "roma_decision_table": str(roma_decision_table_path),
        "roma_bootstrap_summary": str(roma_bootstrap_summary_path),
    },
    "primary_policies": {
        "primary_roma_policy": PRIMARY_ROMA_POLICY,
        "primary_aurora_policy": PRIMARY_AURORA_POLICY,
        "validation_selected_aurora_policy": VALIDATION_SELECTED_AURORA_POLICY,
    },
    "strict_test_period": {
        "n_days": int(len(unified_mat)),
        "start_date": str(unified_mat.index.min().date()),
        "end_date": str(unified_mat.index.max().date()),
    },
    "main_conclusion": (
        "ROMA is best interpreted as a leakage-controlled regime-template baseline. "
        "It did not outperform the B12 moving-average timing benchmark. "
        "AURORA10-UAMV-B is the final uncertainty-aware allocation method and is best interpreted "
        "as a downside-risk control layer rather than a total-return-maximizing strategy."
    ),
    "claim_boundary": [
        "Supported: regime-template allocation alone is insufficient under the tested protocol.",
        "Supported: AURORA10-UAMV-B improves risk-adjusted and downside-risk behavior relative to ROMA.",
        "Supported: AURORA's strongest contribution is downside-risk control.",
        "Not supported: total-return dominance over all benchmarks.",
        "Not supported: guaranteed future investment performance.",
    ],
    "output_paths": {
        "run_root": str(RUN_ROOT),
        "tables": str(TABLE_RUN_DIR),
        "figures": str(PAPER_FIGURE_DIR),
        "manuscript_assets": str(MANUSCRIPT_RUN_DIR),
        "reports": str(REPORT_RUN_DIR),
    },
    "educational_note": (
        "This notebook is for reproducible financial machine-learning research only. "
        "It does not provide personalized financial advice or performance guarantees."
    ),
}

validation_report_path = REPORT_RUN_DIR / "NOTEBOOK13_validation_report.json"
validation_report_global_path = REPORT_DIR / f"NOTEBOOK13_validation_report_{RUN_ID}.json"

save_json(validation_report_path, validation_report)
save_json(validation_report_global_path, validation_report)

manifest_df = make_file_manifest(RUN_ROOT)

manifest_path = REPORT_RUN_DIR / "NOTEBOOK13_file_manifest_SHA256.csv"
manifest_global_path = REPORT_DIR / f"NOTEBOOK13_file_manifest_SHA256_{RUN_ID}.csv"

manifest_df.to_csv(manifest_path, index=False)
manifest_df.to_csv(manifest_global_path, index=False)

# ============================================================
# 18. Final summary
# ============================================================

print("\n" + "=" * 80)
print("NOTEBOOK 13 COMPLETE")
print("=" * 80)
print("Run ID                         :", RUN_ID)
print("Run root                       :", RUN_ROOT)
print("Unified strict-test matrix     :", unified_matrix_path)
print("Paper performance table        :", TABLE_RUN_DIR / "notebook13_paper_focused_performance_table.csv")
print("Rounded performance table      :", TABLE_RUN_DIR / "notebook13_final_compact_performance_table_rounded.csv")
print("Statistical evidence summary   :", TABLE_RUN_DIR / "notebook13_statistical_evidence_summary.csv")
print("Claim checklist                :", TABLE_RUN_DIR / "notebook13_claim_checklist.csv")
print("Methodology progression        :", TABLE_RUN_DIR / "notebook13_methodology_progression.csv")
print("Figure index                   :", TABLE_RUN_DIR / "notebook13_figure_index.csv")
print("Paper figures                  :", PAPER_FIGURE_DIR)
print("Manuscript assets              :", MANUSCRIPT_RUN_DIR)
print("Output index                   :", TABLE_RUN_DIR / "notebook13_output_index.csv")
print("Validation report              :", validation_report_path)
print("Manifest                       :", manifest_path)
print("=" * 80)

print("\nFinal paper-safe conclusion:")
print(
    "ROMA is a leakage-controlled regime-template baseline that did not outperform the strongest timing benchmark. "
    "AURORA10-UAMV-B is the final uncertainty-aware allocation method. "
    "The strongest supported contribution is downside-risk control, not total-return dominance."
)

Mounted at /content/drive
Notebook 13: Unified ROMA-AURORA Final Comparison
Timestamp UTC            : 2026-06-25T06:48:02Z
Run ID                   : 20260625_064802
ROMA R3 run ID           : 20260625_031440
Notebook13 input index   : /content/drive/MyDrive/AURORA_TWETF/outputs/ROMA_TWETF/statistical_significance_block_bootstrap/run_20260625_031440/NOTEBOOK13_ROMA_AURORA_INPUT_INDEX.csv
AURORA Notebook10 root   : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/uncertainty_aware_mean_variance_allocation/run_20260624_100748
AURORA Notebook11 root   : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/statistical_significance_block_bootstrap/run_20260624_124834
AURORA Notebook12 root   : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/paper_tables_figures_manuscript_assets/run_20260624_142307
Run root                 : /content/drive/MyDrive/AURORA_TWETF/outputs/ROMA_AURORA_TWETF/unified_paper_comparison/run_20260625_064802

Step 1: Loading Notebook 13 input inde